In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "paths.py").exists())
sys.path.insert(0, str(ROOT))
from paths import *


## GPU and python


In [ ]:
# OneTrainer on Colab, FLUX.2.
# run the setup cells once per session, then the train cell.
#
# two things that broke this before: a second OneTrainer clone in /content
# shadowing the Drive repo, and installing diffusers from master instead of
# the pinned commit. don't re-add a cell that clones either into /content.
!nvidia-smi

import sys
print("python:", sys.version)
if sys.version_info < (3, 10) or sys.version_info >= (3, 13):
    print("onetrainer wants python >=3.10,<3.13")
else:
    print("python ok")


Wed Aug 26 22:22:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             50W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
import os

PROJECT = str(ROOT)
REPO = f'{PROJECT}/OneTrainer'

os.makedirs(PROJECT, exist_ok=True)

if not os.path.exists(REPO):
    %cd $PROJECT
    !git clone --recursive https://github.com/Nerogar/OneTrainer.git

%cd $REPO

!rm -rf /content/OneTrainer /content/diffusers

!git log -1 --format="OneTrainer commit: %h  %cd"

print("cwd:", os.getcwd())


Mounted at /content/drive
/content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer
OneTrainer commit: 9a270603  Sun Feb 8 09:18:39 2026 +0100
Active dir: /content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer


## Install pinned requirements


In [ ]:
# exact versions this commit wants, pinned diffusers included. restart the runtime after (Runtime > Restart session), re-run the mount cell.
%cd {ROOT}/OneTrainer
!pip install -r requirements.txt


/content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu128
Ignoring triton-windows: markers 'sys_platform == "win32"' don't match your environment
Obtaining diffusers from git+https://github.com/huggingface/diffusers.git@6a1904e#egg=diffusers (from -r requirements-global.txt (line 23))
  Updating ./src/diffusers clone (to revision 6a1904e)
  Running command git fetch -q --tags
  Running command git reset --hard -q 6a1904e
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
Obtaining mgds from git+https://github.com/Nerogar/mgds.git@a0c84a3#egg=mgds (from -r requirements-global.txt (line 35))
  Updating ./src/mgds clone (to revision a0c84a3)
  Running command git fetch -q --tags
  Running command git reset --hard -q a0c84a3
  Installing build dependen

## Verify


In [ ]:
!python -c "import torch, torchvision, transformers, diffusers; print('torch', torch.__version__); print('torchvision', torchvision.__version__); print('transformers', transformers.__version__); print('diffusers', diffusers.__version__)"
!python -c "import torch; print('cuda:', torch.cuda.is_available())"
!python -c "from torchvision.io import write_video; print('write_video OK')"
!python -c "import mgds; print('mgds:', mgds.__file__)"


torch 2.8.0+cu128
torchvision 0.23.0+cu128
transformers 4.56.2
diffusers 0.37.0.dev0
cuda: True
write_video OK
mgds: None


## Check the config


In [ ]:

import json, os

CONFIG = str(ROOT / "pixartsigma_colab.json")
REPO = str(ROOT / "OneTrainer")

c = json.load(open(CONFIG))

for k in ["base_model_name", "model_type", "training_method", "peft_type",
          "train_device", "temp_device", "resolution", "epochs", "batch_size",
          "learning_rate", "save_every", "save_every_unit",
          "workspace_dir", "output_model_destination"]:
    print(f"{k}: {c.get(k)}")

print()
problems = []

if c.get("concepts") == []:
    problems.append("concepts is [] -- must be null, or the concept file is ignored "
                    "and training silently runs on zero images")
if c.get("train_device") != "cuda":
    problems.append(f"train_device is {c.get('train_device')!r} -- should be 'cuda'")
if c.get("tensorboard"):
    problems.append("tensorboard is true -- Colab has no /usr/bin/tensorboard, set false")

cf = c.get("concept_file_name")
if cf:
    resolved = cf if os.path.isabs(cf) else os.path.join(REPO, cf)
    print("concept file:", resolved, "| exists:", os.path.exists(resolved))
    if os.path.exists(resolved):
        for con in json.load(open(resolved)):
            d = con.get("path")
            n = len(os.listdir(d)) if d and os.path.isdir(d) else 0
            print(f"  concept path: {d} | files: {n}")
            if n == 0:
                problems.append(f"concept path has no files: {d}")
    else:
        problems.append("concept file not found")

print()
if problems:
    for p in problems:
        print("PROBLEM:", p)
else:
    print("config looks OK")


base_model_name: PixArt-alpha/PixArt-Sigma-XL-2-1024-MS
model_type: PIXART_SIGMA
training_method: LORA
peft_type: LORA
train_device: cuda
temp_device: cpu
resolution: 1024
epochs: 100
batch_size: 4
learning_rate: 0.0001
save_every: 0
save_every_unit: NEVER
workspace_dir: /content/drive/MyDrive/Synthetic_Plants_Project/workspace/pixart_achillea_run
output_model_destination: /content/drive/MyDrive/Synthetic_Plants_Project/outputs/pixart_achillea/lora.safetensors

concept file: /content/drive/MyDrive/Synthetic_Plants_Project/Notebooks/OneTrainer/modelconfigs/train_concepts.json | exists: True
  concept path: /content/drive/MyDrive/Synthetic_Plants_Project/Datasets/Achillea_Maritima_2 | files: 276

config looks OK


## Train


In [ ]:
# the shim only patches torchvision.io.write_video, which some builds no
# longer export. if the verify cell said write_video OK, skip both shim
# cells and call scripts/train.py directly. /content is wiped on restart,
# so re-run the writefile cell each session.
%%writefile /content/shim.py
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"

import sys, runpy
import torch
torch._dynamo.config.suppress_errors = True

import torchvision.io
if not hasattr(torchvision.io, "write_video"):
    torchvision.io.write_video = lambda *a, **k: None

root = os.getcwd()
sys.path.insert(0, os.path.join(root, "scripts"))
sys.path.insert(0, root)

sys.argv = sys.argv[1:]
runpy.run_path(os.path.join(root, "scripts", "train.py"), run_name="__main__")


Overwriting /content/shim.py


In [ ]:
%cd {ROOT}/OneTrainer
!TORCHDYNAMO_DISABLE=1 python -u /content/shim.py train.py --config-path {ROOT}/training/configs/flux2/flux2_colab_achillea.json


In [ ]:
%cd {ROOT}/OneTrainer
!python -u /content/shim.py train.py --config-path {ROOT}/training/configs/flux2/flux2_colab_achillea.json


Streaming output truncated to the last 5000 lines.
step:  33% 22/67 [00:48<01:33,  2.08s/it, loss=0.832, smooth loss=0.954]
step:  34% 23/67 [00:48<01:31,  2.08s/it, loss=0.832, smooth loss=0.954]
step:  34% 23/67 [00:50<01:31,  2.08s/it, loss=0.777, smooth loss=0.953]
step:  36% 24/67 [00:50<01:29,  2.08s/it, loss=0.777, smooth loss=0.953]
step:  36% 24/67 [00:52<01:29,  2.08s/it, loss=0.957, smooth loss=0.953]
step:  37% 25/67 [00:52<01:27,  2.08s/it, loss=0.957, smooth loss=0.953]
step:  37% 25/67 [00:54<01:27,  2.08s/it, loss=0.808, smooth loss=0.951]
step:  39% 26/67 [00:54<01:25,  2.09s/it, loss=0.808, smooth loss=0.951]
step:  39% 26/67 [00:56<01:25,  2.09s/it, loss=0.793, smooth loss=0.95] 
step:  40% 27/67 [00:56<01:23,  2.09s/it, loss=0.793, smooth loss=0.95]
step:  40% 27/67 [00:58<01:23,  2.09s/it, loss=1.08, smooth loss=0.951]
step:  42% 28/67 [00:58<01:21,  2.09s/it, loss=1.08, smooth loss=0.951]
step:  42% 28/67 [01:00<01:21,  2.09s/it, loss=1.18, smooth loss=0.953]
step

In [ ]:
%cd {ROOT}/OneTrainer
!python -u /content/shim.py train.py --config-path {ROOT}/training/configs/flux2/flux2_colab_eryngium.json


Streaming output truncated to the last 5000 lines.
step:  16% 7/44 [00:16<01:17,  2.09s/it, loss=0.852, smooth loss=0.96]
step:  18% 8/44 [00:16<01:15,  2.09s/it, loss=0.852, smooth loss=0.96]
step:  18% 8/44 [00:18<01:15,  2.09s/it, loss=0.725, smooth loss=0.957]
step:  20% 9/44 [00:18<01:13,  2.09s/it, loss=0.725, smooth loss=0.957]
step:  20% 9/44 [00:20<01:13,  2.09s/it, loss=0.831, smooth loss=0.956]
step:  23% 10/44 [00:20<01:11,  2.09s/it, loss=0.831, smooth loss=0.956]
step:  23% 10/44 [00:22<01:11,  2.09s/it, loss=0.941, smooth loss=0.956]
step:  25% 11/44 [00:22<01:09,  2.09s/it, loss=0.941, smooth loss=0.956]
step:  25% 11/44 [00:25<01:09,  2.09s/it, loss=1, smooth loss=0.956]    
step:  27% 12/44 [00:25<01:06,  2.09s/it, loss=1, smooth loss=0.956]
step:  27% 12/44 [00:27<01:06,  2.09s/it, loss=0.873, smooth loss=0.956]
step:  30% 13/44 [00:27<01:04,  2.09s/it, loss=0.873, smooth loss=0.956]
step:  30% 13/44 [00:29<01:04,  2.09s/it, loss=1.02, smooth loss=0.956] 
step:  32% 

In [ ]:
%cd {ROOT}/OneTrainer
!python -u /content/shim.py train.py --config-path {ROOT}/training/configs/flux2/flux2_colab_carpobrotus.json


Streaming output truncated to the last 5000 lines.
step:  85% 28/33 [01:01<00:10,  2.11s/it, loss=0.765, smooth loss=0.918]
step:  88% 29/33 [01:01<00:08,  2.11s/it, loss=0.765, smooth loss=0.918]
step:  88% 29/33 [01:03<00:08,  2.11s/it, loss=0.943, smooth loss=0.918]
step:  91% 30/33 [01:03<00:06,  2.11s/it, loss=0.943, smooth loss=0.918]
step:  91% 30/33 [01:05<00:06,  2.11s/it, loss=0.69, smooth loss=0.916] 
step:  94% 31/33 [01:05<00:04,  2.11s/it, loss=0.69, smooth loss=0.916]
step:  94% 31/33 [01:07<00:04,  2.11s/it, loss=0.894, smooth loss=0.916]
step:  97% 32/33 [01:07<00:02,  2.11s/it, loss=0.894, smooth loss=0.916]
step:  97% 32/33 [01:09<00:02,  2.11s/it, loss=0.953, smooth loss=0.916]
step: 100% 33/33 [01:09<00:00,  2.11s/it, loss=0.953, smooth loss=0.916]
epoch:  29% 29/100 [43:03<1:31:02, 76.94s/it]
step:   0% 0/33 [00:00<?, ?it/s]
step:   0% 0/33 [00:02<?, ?it/s, loss=0.95, smooth loss=0.916]
step:   3% 1/33 [00:02<01:07,  2.11s/it, loss=0.95, smooth loss=0.916]
step:  

In [1]:
from safetensors import safe_open
import os
BASE = str(ROOT)
for m in ["sdxl", "flux2", "qwen"]:
    p = f"{BASE}/outputs/{m}_achillea/lora.safetensors"
    if not os.path.exists(p):
        print(f"{m}: not trained yet"); continue
    with safe_open(p, framework="pt") as f:
        keys = list(f.keys())
    print(f"\n{m}: {len(keys)} tensors")
    for k in keys[:6]:
        print("  ", k)


sdxl: not trained yet
flux2: not trained yet
qwen: not trained yet


In [8]:
import json, os

BASE = str(ROOT)

# correct the model id in the script
p = f"{BASE}/Configs_and_Concepts/generate_evalflux.py"
src = open(p).read()
src = src.replace('"black-forest-labs/FLUX.2-dev"',
                  '"black-forest-labs/FLUX.2-klein-base-9B"')
open(p, "w").write(src)

# pull the token OneTrainer used
cfg = json.load(open(f"{BASE}/Configs_and_Concepts/flux2_lora/flux2_colab_achillea.json"))
tok = cfg["secrets"]["huggingface_token"]

from huggingface_hub import login
login(token=tok, add_to_git_credential=False)
print("logged in; model id now:",
      [l for l in open(p) if "klein" in l][0].strip())


logged in; model id now: "model_id": "black-forest-labs/FLUX.2-klein-base-9B",


In [9]:
%cd {ROOT}
!python Configs_and_Concepts/generate_evalflux.py --model flux2 --all --n 100


/content/drive/MyDrive/Synthetic_Plants_Project
model=flux2  n=100  species: achillea, eryngium, carpobrotus

flux2/achillea
  prompt: a detailed realistic photograph of l35tgyhtew
  size:   512x512  steps=28  cfg=3.5
  out:    /content/drive/MyDrive/Synthetic_Plants_Project/generated/flux2_achillea
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
[transformers] `Siglip2ImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Siglip2ImageProcessor` instead.
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  

In [ ]:
%cd {ROOT}/OneTrainer
!python -u /content/shim.py train.py --config-path {ROOT}/training/configs/qwen/qwen_colab_carpobrotus.json


In [ ]:
%cd {ROOT}/OneTrainer
!python -u /content/shim.py train.py --config-path {ROOT}/training/configs/pixart/pixartsigma_colab_carpobrotus.json


In [ ]:
%cd {ROOT}/OneTrainer
!python -u /content/shim.py train.py --config-path {ROOT}/training/configs/pixart/pixartsigma_colab_carpobrotus.json


In [ ]:
%cd {ROOT}
!python Configs_and_Concepts/generate_eval.py --all --n 100


In [ ]:
# @title
%cd {ROOT}
!python Configs_and_Concepts/generate_eval.py --species achillea --n 2


/content/drive/MyDrive/Synthetic_Plants_Project
2026-08-26 20:52:04.171909: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-26 20:52:04.243580: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
generating 2 per species: ac

In [ ]:
# @title
from safetensors import safe_open
p = str(ROOT / "outputs/lora/pixart_achillea/lora.safetensors")
with safe_open(p, framework="pt") as f:
    keys = list(f.keys())
print(len(keys), "tensors")
for k in keys[:12]:
    print(" ", k)


861 tensors
  lora_transformer_adaln_single_emb_timestep_embedder_linear_1.alpha
  lora_transformer_adaln_single_emb_timestep_embedder_linear_1.lora_down.weight
  lora_transformer_adaln_single_emb_timestep_embedder_linear_1.lora_up.weight
  lora_transformer_adaln_single_emb_timestep_embedder_linear_2.alpha
  lora_transformer_adaln_single_emb_timestep_embedder_linear_2.lora_down.weight
  lora_transformer_adaln_single_emb_timestep_embedder_linear_2.lora_up.weight
  lora_transformer_adaln_single_linear.alpha
  lora_transformer_adaln_single_linear.lora_down.weight
  lora_transformer_adaln_single_linear.lora_up.weight
  lora_transformer_caption_projection_linear_1.alpha
  lora_transformer_caption_projection_linear_1.lora_down.weight
  lora_transformer_caption_projection_linear_1.lora_up.weight


## Backup Resume Cell


In [ ]:
# @title
# !ls -la {ROOT}/outputs/workspace*/backup/

# %cd {ROOT}/OneTrainer
# !python -u /content/shim.py train.py \
#   --config-path {ROOT}/training/configs/pixart/pixartsigma_colab.json \
#   --resume-from-checkpoint {ROOT}/outputs/workspace<run>/backup/last


## Freeze the environment


In [ ]:
# @title
from datetime import datetime
stamp = datetime.now().strftime("%Y%m%d_%H%M")
!pip freeze > {ROOT}/requirements_frozen_{stamp}.txt
print("written: requirements_frozen_" + stamp + ".txt")


written: requirements_frozen_20260825_0018.txt
